# join_optimization-tpch-spark-iceberg

## 1. Overview

## 2. Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.remote("sc://spark-connect:15002").getOrCreate()

## 3. Read

In [ ]:
orders = spark.read.parquet("s3a://landing/tpch/orders")
customer = spark.read.parquet("s3a://landing/tpch/customer")

## 4. Transform

In [ ]:
joined = orders.join(F.broadcast(customer), F.col("o_custkey") == F.col("c_custkey"))
joined.explain()
mart = joined.groupBy("c_mktsegment").agg(F.sum("o_totalprice").alias("revenue"), F.count("*").alias("orders"))

## 5. Write

In [ ]:
mart.writeTo("lakehouse.gold.tpch_segment_revenue").using("iceberg").createOrReplace()

## 6. Verify

In [ ]:
print("AQE:", spark.conf.get("spark.sql.adaptive.enabled"))
spark.table("lakehouse.gold.tpch_segment_revenue").orderBy(F.desc("revenue")).show(truncate=False)